In [1]:
import gc
import json
import random
from pathlib import Path

import cv2
import shutil
import time
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet34_Weights, resnet34

**Load selected checkpoint**

In [4]:
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "vision_unit_02_outputs/"
    "block_03"
)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_VERSION = "v1"

experiment_summary = pd.read_csv(OUTPUT_ROOT/ "experiment_summary.csv")

### U-Net Model

**Convolution and decoder blocks**

In [5]:
class DoubleConv(nn.Module):
    def __init__(self, input_channels, output_channels):
        super().__init__()
        
        self.block = nn.Sequential(
            nn.Conv2d(
                input_channels, output_channels,
                kernel_size=3, padding=1, bias=False
            ),
            nn.BatchNorm2d(output_channels),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(output_channels, output_channels,
                      kernel_size=3, padding=1, bias=False
            ),
            nn.BatchNorm2d(output_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.block(x)

In [6]:
class DecoderBlock(nn.Module):
    def __init__(self, input_channels, skip_channels, output_channels):
        super().__init__()
        
        self.convolution = DoubleConv(input_channels + skip_channels, output_channels)
    
    def forward(self, inputs, skip):
        inputs = F.interpolate(
            inputs, size=skip.shape[-2:],
            mode="bilinear", align_corners=False
        )
        combined = torch.cat([inputs, skip], dim=1)
        return self.convolution(combined)

In [7]:
class ResNet34UNet(nn.Module):
    def __init__(self, weights=None):
        super().__init__()
        
        backbone = resnet34(weights=weights)
        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu
        )
        self.pool = backbone.maxpool
        
        self.encoder1 = backbone.layer1
        self.encoder2 = backbone.layer2
        self.encoder3 = backbone.layer3
        self.encoder4 = backbone.layer4
        
        self.decoder_4 = DecoderBlock(
            input_channels=512, skip_channels=256, output_channels=256
        )
        self.decoder_3 = DecoderBlock(
            input_channels=256, skip_channels=128, output_channels=128
        )
        self.decoder_2 = DecoderBlock(
            input_channels=128, skip_channels=64, output_channels=64
        )
        self.decoder_1 = DecoderBlock(
            input_channels=64, skip_channels=64, output_channels=32
        )
        
        self.final_refinement = DoubleConv(
            input_channels=32, output_channels=32
        )
        
        self.segmentation_head = nn.Conv2d(32, 1, kernel_size=1)


    def forward(self, inputs):
        input_size = inputs.shape[-2:]
        
        skip_0 = self.stem(inputs)
        skip_1 = self.encoder1(self.pool(skip_0))
        skip_2 = self.encoder2(skip_1)
        skip_3 = self.encoder3(skip_2)
        
        bottleneck = self.encoder4(skip_3)
        
        decoded = self.decoder_4(bottleneck, skip_3)
        decoded = self.decoder_3(decoded, skip_2)
        decoded = self.decoder_2(decoded, skip_1)
        decoded = self.decoder_1(decoded, skip_0)
        
        decoded = F.interpolate(
            decoded, size=input_size,
            mode="bilinear", align_corners=False
        )
        
        decoded = self.final_refinement(decoded)
        logits = self.segmentation_head(decoded)
        
        return logits

In [8]:
def load_best_model(checkpoint_path):
    checkpoint = torch.load(
        checkpoint_path, map_location=DEVICE, weights_only=False
    )
    
    loaded_model = ResNet34UNet(weights=None).to(DEVICE)
    loaded_model.load_state_dict(checkpoint["model_state"])
    loaded_model.eval()

    return loaded_model, checkpoint

In [9]:
seleted_row = experiment_summary.sort_values("dice", ascending=False).iloc[0]

SELETED_EXPERIMENT = str(seleted_row["experiment"])
SELECTED_THRESHOLD = str(seleted_row["best_threshold"])

SELECTED_CHECKPOINT_PATH = (
    OUTPUT_ROOT / (
        f"{RUN_VERSION}_"
        f"{SELETED_EXPERIMENT}"
        ) / "best.pt"
    )

In [10]:
seleted_model, selected_checkpoint = load_best_model(SELECTED_CHECKPOINT_PATH)

print( "Experiment:", SELETED_EXPERIMENT)
print("Checkpoint epoch:", selected_checkpoint["epoch"])
print("Threshold:", SELECTED_THRESHOLD)

Experiment: bce_dice
Checkpoint epoch: 2
Threshold: 0.3


**Per-image and boundary metrics**

In [12]:
def assign_mask_size(crack_ratio):
    if crack_ratio <= 0.0:
        return "empty"

    if crack_ratio < 0.005:
        return "tiny_<0.5%"

    if crack_ratio < 0.02:
        return "small_0.5-2%"

    if crack_ratio < 0.10:
        return "medium_2-10%"

    return "large_>=10%"

In [13]:
def calculate_single_mask_metrics(prediction, target):
    prediction = prediction.astype(bool)
    target = target.astype(bool)

    tp = int(np.logical_and(prediction, target).sum())
    fp = int(np.logical_and(prediction, ~target).sum())

    fn = int(np.logical_and( ~prediction,target,).sum())
    tn = int(np.logical_and(~prediction, ~target,).sum())

    precision_denominator = (tp + fp)
    recall_denominator = (tp + fn)

    iou_denominator = (tp + fp + fn)
    dice_denominator = (2 * tp + fp + fn)

    precision = (
        tp / precision_denominator
        if precision_denominator > 0
        else np.nan
    )

    recall = (
        tp / recall_denominator
        if recall_denominator > 0
        else np.nan
    )

    iou = (
        tp / iou_denominator
        if iou_denominator > 0
        else 1.0
    )

    dice = (
        2 * tp / dice_denominator
        if dice_denominator > 0
        else 1.0
    )

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "iou": iou,
        "dice": dice,
    }

In [14]:
def extract_boundary(mask):
    mask = mask.astype(np.uint8)
    kernel = np.ones((3, 3),dtype=np.uint8)
    eroded = cv2.erode(mask, kernel, iterations=1)

    return (mask - eroded).astype(bool)